# Thesis Baseline Setup

This notebook documents the workflow for the thesis baseline on xBD/xView2 using a building-level classification pipeline.

---

## Core Principle

- Preprocessing and model training run directly in Python
- Notebook is used for:
  - Data inspection
  - Sanity checks
  - Training control
  - Evaluation
  - Visualization

The official xView2 Docker pipeline is not used as the main experimental setup, because the thesis isolates the damage classification task from localization by using ground truth building polygons.

---

## Required Data

Download and place on Desktop:

- `train`
- `test`
- `hold`

These contain:
- images
- labels

---

## Main Experimental Design

The thesis does not use the original end to end competition pipeline as the primary method.

Instead, the pipeline is:

1. Read ground truth polygons from label JSON files
2. Build a building-level metadata table
3. Extract pre/post building crops
4. Stack pre and post crops into 6-channel tensors
5. Train a ResNet50-based classifier on damage labels

This removes localization as a confounding factor and isolates the classification task.

---

## Required External Apps

- `miniconda` or another Python environment manager


## Create Environment

```bash
conda create -n thesis_xview python=3.10
conda activate thesis_xview

python --version
which python

python -m pip install --upgrade pip setuptools wheel

python -m pip install -r requirements.txt

python -m ipykernel install --user --name thesis_xview --display-name "Python thesis_xview" '''


## Create Environment

```bash
conda create -n thesis_xview python=3.10
conda activate thesis_xview

python --version
which python

python -m pip install --upgrade pip setuptools wheel

python -m pip install -r requirements.txt

python -m ipykernel install --user --name thesis_xview --display-name "Python thesis_xview"
```

## Restart Kernel

Restart VS Code or Jupyter, then select the new kernel:

```text
Python thesis_xview
```

from:

```text
Kernel → Change Kernel → thesis_xview
```

In [ ]:
import sys
!{sys.executable} -m pip install -r requirements.txt

In [ ]:
import sys
import torch
import torchvision

print(sys.executable)
print(sys.version)
print(torch.__version__)
print(torchvision.__version__)
print("MPS available:", torch.backends.mps.is_available())

# Training, Validation, and Class Weighting Strategy

## 1. Data Splits: Train, Test, and Hold

The dataset is divided into three distinct subsets, each with a specific role in the machine learning pipeline:

### Train Split
The **training set** is used to learn the model parameters. During this phase, the model updates its weights based on the input data and corresponding labels.

### Test Split
The **test set** is used during training to evaluate model performance after each epoch. It serves as a **validation set** for:

- monitoring learning progress  
- selecting the best model (e.g., best epoch)  
- comparing configurations or hyperparameters  

Importantly, the model does **not learn from the test set**. It is only used for evaluation.

### Hold Split
The **holdout set** is used for the **final evaluation** after training is complete. It provides an unbiased estimate of model performance because:

- it is not used during training  
- it is not used for model selection  

This separation ensures that reported results reflect true generalization.

---

### Summary of Roles

| Split | Purpose |
|------|--------|
| Train | Learn model parameters |
| Test | Model selection and monitoring |
| Hold | Final unbiased evaluation |

---

## 2. Class Imbalance and Class Weights

### Problem: Imbalanced Data

In the dataset, the distribution of damage classes is highly imbalanced. For example:

- "no-damage" appears far more frequently  
- "minor", "major", and "destroyed" are much less common  

Without correction, the model would tend to predict the majority class ("no-damage") to minimize loss, leading to poor performance on rare but important damage categories.

---

### Solution: Class-Weighted Loss

To address this, a **class-weighted cross-entropy loss** is used.

Each class is assigned a weight:

```text
[0.3402, 2.6667, 2.8210, 3.0202]

In [10]:
import sys
!{sys.executable} model/classification_baseline.py

Seed 123 | Epoch 4/8: 100%|████| 4994/4994 [31:41<00:00,  2.63it/s, loss=0.4268]
Seed 123 | Epoch 04 | Train Loss: 0.2822 | Val Loss: 0.3968 | Val Macro F1: 0.7034 | Time: 34.30 min
Seed 123 | Epoch 5/8: 100%|████| 4994/4994 [25:36<00:00,  3.25it/s, loss=0.1194]
Seed 123 | Epoch 05 | Train Loss: 0.2138 | Val Loss: 0.4669 | Val Macro F1: 0.6993 | Time: 28.04 min
Seed 123 | Epoch 6/8: 100%|████| 4994/4994 [24:45<00:00,  3.36it/s, loss=0.1564]
Seed 123 | Epoch 06 | Train Loss: 0.1550 | Val Loss: 0.5047 | Val Macro F1: 0.6869 | Time: 27.15 min
Seed 123 | Epoch 7/8: 100%|████| 4994/4994 [24:16<00:00,  3.43it/s, loss=0.3657]
Seed 123 | Epoch 07 | Train Loss: 0.1148 | Val Loss: 0.5725 | Val Macro F1: 0.6806 | Time: 26.60 min
Seed 123 | Epoch 8/8: 100%|████| 4994/4994 [24:19<00:00,  3.42it/s, loss=0.0653]
Seed 123 | Epoch 08 | Train Loss: 0.0926 | Val Loss: 0.5920 | Val Macro F1: 0.6760 | Time: 26.65 min

Starting unweighted baseline seed 999

Using unweighted cross entropy.

Starting training

# Model Performance Analysis and Interpretation

## 1. Training Dynamics

The training process exhibits a clear pattern of convergence followed by overfitting. Training loss decreases consistently across epochs, while validation loss initially decreases and then begins to increase after approximately the second epoch.

The mean macro F1 score on the validation set peaks at **epoch 7 (0.6579 ± 0.0063)**. The 1SD model selection rule also selects **epoch 7**, indicating that this epoch provides the best balance between validation performance and stability across seeds.

This behavior suggests that:

- the model has sufficient capacity to fit the dataset
- prolonged training improves training loss but does not consistently improve validation loss
- model selection based on validation macro F1 is necessary to avoid selecting later overfitted checkpoints

---

## 2. Overall Performance

The final performance of the model is summarized below:

| Split | Macro F1 |
|------|--------|
| Test | 0.6579 ± 0.0063 |
| Hold | 0.6717 ± 0.0057 |

The slightly higher performance on the holdout set indicates that:

- the model generalizes reasonably well beyond the validation split
- there is no evidence of overfitting to the validation split at the selected checkpoint
- the optimized OOD splits are broadly consistent, although performance remains environment dependent

Overall, a macro F1 score in the range of **0.66–0.67** represents a strong baseline for this task using a standard ResNet50 architecture without extensive hyperparameter tuning.

---

## 3. Class-wise Performance

The model exhibits varying performance across damage categories.

### No-damage

- F1 score: high (≈ 0.89–0.91)
- strong precision and moderately lower recall

This is expected due to the high frequency and clear visual characteristics of this class.

---

### Minor and Major Damage

- F1 scores: moderate (minor ≈ 0.52–0.57, major ≈ 0.50–0.52)
- lower precision compared to recall

These classes are more difficult to distinguish due to:

- subtle visual differences
- ambiguity between damage levels
- overlapping feature representations

---

### Destroyed

- F1 score: relatively strong (≈ 0.70–0.71)

This class benefits from more distinct visual features, making it easier for the model to identify.

---

## 4. Precision–Recall Trade-off

A notable pattern emerges across all damage classes:

- **high recall**
- **lower precision**

This indicates that the model tends to **over-predict damage**, classifying uncertain cases as damaged rather than undamaged.

This behavior is directly influenced by the use of class-weighted loss.

---

## 5. Effect of Class Weights

Class weights were applied to address the strong imbalance in the dataset, where the "no-damage" class dominates.

The applied class weights were:

| Class | Weight |
|------|--------|
| No-damage | 0.3402 |
| Minor damage | 2.6668 |
| Major damage | 2.8210 |
| Destroyed | 3.0202 |

By assigning higher weights to minority classes, the model is encouraged to prioritize correct classification of damaged buildings.

This leads to:

### Positive effects

- improved recall for damage classes  
- better detection of rare events  
- more balanced class performance  

### Side effects

- increased false positives for damage  
- confusion between adjacent damage levels  
- reduced precision  

This trade-off is expected and reflects a deliberate modeling choice to prioritize detection of damage over conservative classification.

---

## 6. Generalization to Holdout Data

The model performs slightly better on the holdout set than on the validation set.

This suggests that:

- the model does not overfit the validation process  
- learned features generalize across the held-out split  
- the evaluation protocol is reasonably robust across random seeds  

However, the worst holdout environment is **mexico-earthquake**, with a mean macro F1 of **0.2490 ± 0.0035**. This indicates that while average holdout performance is strong, generalization is not uniform across environments.

This is an important indicator of the remaining difficulty of location-based OOD generalization.

---

## 7. Limitations

Despite strong baseline performance, several limitations remain:

- difficulty distinguishing between intermediate damage levels  
- sensitivity to class imbalance  
- overfitting after early epochs  
- substantial variation across disaster environments  
- weak worst-environment performance, especially for mexico-earthquake  
- reliance on bounding box crops without fine-grained localization  

These limitations are consistent with known challenges in post-disaster damage classification tasks.

These findings provide a solid foundation for further experimentation and model refinement.

## Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pathlib import Path

# =========================
# Configuration
# =========================

split = "hold"  # "val" or "hold"
seed = 42       # 42, 123, or 999

labels = ["no-damage", "minor-damage", "major-damage", "destroyed"]

BASE_PATH = (
    Path.home()
    / "Desktop"
    / "training_outputs"
    / "baseline_resnet50_3seeds_1sd"
    / f"seed_{seed}"
)

y_true_path = BASE_PATH / f"{split}_targets_selected_1sd.npy"
y_pred_path = BASE_PATH / f"{split}_preds_selected_1sd.npy"

if not y_true_path.exists() or not y_pred_path.exists():
    raise FileNotFoundError(
        f"Could not find prediction files.\n\n"
        f"Expected:\n{y_true_path}\n{y_pred_path}"
    )

print(f"Using prediction files from:\n{BASE_PATH}")

# =========================
# Load predictions
# =========================

y_true = np.load(y_true_path)
y_pred = np.load(y_pred_path)

# =========================
# Raw confusion matrix
# =========================

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, values_format="d", colorbar=True)

plt.title(f"{split.upper()} Confusion Matrix - Seed {seed} (Counts)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# =========================
# Normalized confusion matrix
# =========================

cm_norm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    normalize="true",
)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=labels)
disp.plot(ax=ax, values_format=".2f", colorbar=True)

plt.title(f"{split.upper()} Confusion Matrix - Seed {seed} (Normalized)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Confusion Matrix Analysis

The normalized confusion matrix provides insight into the model’s class-wise prediction behavior.

### Overall Performance

The model shows strong performance along the diagonal, indicating good classification accuracy across all classes. In particular:

- **no-damage** achieves the highest accuracy (**0.85**)
- **destroyed** also performs strongly (**0.78**)
- **minor-damage** and **major-damage** show lower accuracy (**0.64** and **0.58**)

This confirms that the model performs best on visually distinct classes, while intermediate damage levels remain more challenging.

---

### Error Patterns

The main misclassifications occur between adjacent damage levels:

- **minor-damage → no-damage (0.16)**
- **minor-damage → major-damage (0.16)**
- **major-damage → minor-damage (0.18)**
- **major-damage → no-damage (0.17)**

This indicates substantial difficulty in distinguishing intermediate levels of structural damage, likely due to subtle visual differences and overlapping feature representations.

Additionally:

- some **no-damage** instances are predicted as **minor-damage (0.08)**
- some **major-damage** instances are predicted as **destroyed (0.08)**
- some **destroyed** instances are predicted as **major-damage (0.08)**

These errors suggest a tendency toward confusion between neighboring severity categories rather than completely random misclassification.

---

### Model Bias

The confusion matrix reveals a consistent pattern:

- relatively high recall for damaged classes  
- increased confusion between adjacent damage categories  
- moderate over-prediction of damage severity in uncertain cases  

This behavior aligns with the use of **class-weighted loss**, which encourages the model to prioritize detecting damage over conservative predictions.

At the same time, the confusion between **minor-damage** and **major-damage** highlights the inherent ambiguity of fine-grained damage classification from satellite imagery.

---

### Summary

The confusion matrix confirms that:

- the model effectively distinguishes between **no damage** and **severe destruction**
- most errors occur between **adjacent damage categories**
- intermediate damage levels remain the most difficult classes
- the model exhibits a moderate bias toward predicting damaged categories

These findings are consistent with the overall performance metrics and highlight the inherent difficulty of fine-grained post-disaster damage classification.

## Sanity Check: Verification of ResNet50-Based Architecture

Before proceeding with full training and experimentation, a structural sanity check was performed to ensure that the implemented model correctly corresponds to a ResNet50 architecture adapted to the specific task.


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# =========================
# 6-channel ResNet50
# =========================

class ResNet50SixChannel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # Load pretrained backbone
        weights = ResNet50_Weights.IMAGENET1K_V2
        self.backbone = resnet50(weights=weights)

        # Original first convolution
        old_conv = self.backbone.conv1

        # Replace with 6-channel convolution
        new_conv = nn.Conv2d(
            in_channels=6,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )

        # Initialize new channels from pretrained RGB weights
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = old_conv.weight
            new_conv.weight[:, 3:, :, :] = old_conv.weight

        self.backbone.conv1 = new_conv

        # Replace classification head
        self.backbone.fc = nn.Linear(
            self.backbone.fc.in_features,
            num_classes,
        )

    def forward(self, x):
        return self.backbone(x)

# =========================
# Instantiate model
# =========================

model = ResNet50SixChannel(num_classes=4)

# =========================
# Sanity checks
# =========================

print("\nFirst convolution:")
print(model.backbone.conv1)

print("\nClassifier head:")
print(model.backbone.fc)

# Verify input channels
assert model.backbone.conv1.in_channels == 6

# Verify output classes
assert model.backbone.fc.out_features == 4

# Dummy forward pass
x = torch.randn(2, 6, 224, 224)

with torch.no_grad():
    y = model(x)

print("\nForward pass output shape:")
print(y.shape)

assert y.shape == (2, 4)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nSanity check passed.")

The structural sanity check confirms that the implemented model correctly corresponds to a ResNet50-based architecture adapted for the task.

### Key Results

- **Input Layer:**  
  `Conv2d(6, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)`  
  → Confirms successful adaptation from 3 to **6 input channels**, enabling joint processing of pre- and post-disaster imagery.

- **Output Layer:**  
  `Linear(in_features=2048, out_features=4, bias=True)`  
  → Confirms correct modification of the classifier head for **4 damage classes**, while preserving the standard ResNet50 feature dimension.

- **Forward Pass Output:**  
  `torch.Size([2, 4])`  
  → Confirms that the network produces the expected output dimensionality for batch-based multi-class classification.

- **Parameter Count:**  
  `23,525,636 trainable parameters`  
  → Consistent with a **ResNet50-scale model**, accounting for the additional parameters introduced by the 6-channel input layer and the modified classifier head.

### Interpretation

These results verify that:

- the backbone architecture remains **ResNet50**
- the model is correctly configured for the **6-channel input representation**
- the classification head matches the **task-specific output space**
- the forward propagation pipeline functions correctly without tensor-shape inconsistencies
- all parameters remain trainable during fine-tuning

### Conclusion

The sanity check confirms that the model is structurally valid and correctly adapted, ensuring that subsequent training and evaluation are performed on a properly specified ResNet50-based architecture.